In [0]:
-- 步骤1: 创建临时视图读取Bronze数据
-- 获取 ADF 传递的参数用新方法
-- DECLARE tdspath STRING DEFAULT getArgument('p_tdspath', 'gmall/order_info/default');
CREATE OR REPLACE TEMPORARY VIEW bronze_user_info_raw AS
SELECT 
    *,
    _metadata.file_path as source_file,
    current_timestamp() as load_timestamp
FROM read_files('abfss://bronze@sahyivy.dfs.core.windows.net/'||:p_tdspath||'/*.parquet');

-- 步骤2: 合并新数据到Silver表（UPSERT）
MERGE INTO silver_user_info AS target
USING bronze_user_info_raw AS source
ON target.id = source.id AND target.is_latest = true
    WHEN MATCHED AND (
    MD5(CONCAT(target.id, target.name, target.phone_num, target.email,target.user_level)) <> md5(CONCAT(source.id, source.name, source.phone_num, source.email,source.user_level))
) THEN
        UPDATE SET                
                target.is_latest = false,
                target.update_timestamp = current_timestamp()
    WHEN NOT MATCHED THEN
        INSERT (id          ,
                login_name            ,
                nick_name             ,
                passwd                ,
                name                  ,
                phone_num             ,
                email                 ,
                head_img              ,
                user_level            ,
                birthday              ,
                gender                ,
                create_time           ,
                operate_time          ,
                status                ,
                source_file           ,
                load_timestamp        ,
                update_timestamp    ,
                is_latest
                )
        VALUES (source.id          ,
                source.login_name            ,
                source.nick_name             ,
                source.passwd                ,
                source.name                  ,
                source.phone_num             ,
                source.email                 ,
                source.head_img              ,
                source.user_level            ,
                source.birthday              ,
                source.gender                ,
                source.create_time           ,
                source.operate_time          ,
                source.status                ,
                source.source_file           , 
                source.load_timestamp        ,
                current_timestamp()        ,
                true
                );